In [ ]:
# XBRL Analysis and Validation Notebook
# Part 11: Cross-validate PDF extractions with XBRL data

import sys
import os
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from xbrl_extraction_system import XBRLValidationSystem, XBRLDownloader, XBRLParser, PDFXBRLValidator

# ================================
# CELL 1: Setup and Configuration
# ================================

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("XBRL Analysis and Validation Notebook")
print("=" * 50)
print("This notebook demonstrates:")
print("1. Downloading XBRL files from SEC EDGAR")
print("2. Parsing XBRL financial data")
print("3. Comparing with PDF-extracted tables")
print("4. Generating validation reports")

# ================================
# CELL 2: Configuration Parameters
# ================================

# Configuration
TICKER = "AAPL"  # Apple Inc.
FILING_TYPE = "10-K"
YEAR = 2023
OUTPUT_DIR = "../data/xbrl_validation"

print(f"Analysis Configuration:")
print(f"  Ticker: {TICKER}")
print(f"  Filing Type: {FILING_TYPE}")
print(f"  Year: {YEAR}")
print(f"  Output Directory: {OUTPUT_DIR}")

# ================================
# CELL 3: Initialize XBRL System
# ================================

# Initialize the XBRL validation system
validator_system = XBRLValidationSystem(OUTPUT_DIR)

print("Initialized XBRL Validation System")
print("Components:")
print("  - XBRLDownloader: Download XBRL files from SEC")
print("  - XBRLParser: Parse financial data from XBRL")
print("  - PDFXBRLValidator: Compare with PDF extractions")

# ================================
# CELL 4: Download XBRL Files
# ================================

print(f"Downloading XBRL files for {TICKER} {FILING_TYPE} {YEAR}...")

# Download XBRL files
xbrl_files = validator_system.downloader.download_filing_xbrl(
    ticker=TICKER, 
    filing_type=FILING_TYPE, 
    count=1, 
    year=YEAR
)

if xbrl_files:
    print(f"Downloaded {len(xbrl_files)} XBRL files:")
    for file in xbrl_files:
        print(f"  - {file}")
        print(f"    Size: {file.stat().st_size / 1024:.1f} KB")
else:
    print("No XBRL files found. Check ticker symbol and year.")

# ================================
# CELL 5: Parse XBRL Data
# ================================

if xbrl_files:
    print(f"Parsing XBRL file: {xbrl_files[0].name}")
    
    # Parse the XBRL file
    xbrl_data = validator_system.parser.parse_xbrl_file(xbrl_files[0])
    
    if xbrl_data:
        print(f"XBRL parsing completed using: {xbrl_data['parsing_method']}")
        print(f"Found {len(xbrl_data['raw_facts'])} raw facts")
        print(f"Found {len(xbrl_data['contexts'])} contexts")
        
        # Display financial concepts found
        print("\nFinancial concepts found:")
        for concept, facts in xbrl_data['financial_data'].items():
            if facts:
                print(f"  {concept}: {len(facts)} facts")
    else:
        print("Failed to parse XBRL data")

# ================================
# CELL 6: Extract Key Financial Values
# ================================

if xbrl_files and xbrl_data:
    # Get latest annual values
    annual_values = validator_system.parser.get_latest_annual_values(xbrl_data['financial_data'])
    
    print("Latest Annual Financial Values from XBRL:")
    print("=" * 50)
    
    if annual_values:
        # Create DataFrame for better display
        xbrl_df = pd.DataFrame([
            {"Financial Concept": concept, "XBRL Value ($M)": f"${value/1000000:.1f}M"}
            for concept, value in annual_values.items()
        ])
        
        print(xbrl_df.to_string(index=False))
    else:
        print("No annual values found in XBRL data")

# ================================
# CELL 7: Load PDF-Extracted Tables
# ================================

print(f"\nLoading PDF tables for {TICKER}_{FILING_TYPE}_{YEAR}...")

# Load PDF tables
pdf_tables = validator_system.validator.load_pdf_tables(f"{TICKER}_{FILING_TYPE}_{YEAR}")

if pdf_tables:
    print(f"Loaded {len(pdf_tables)} PDF tables:")
    for table_name, df in pdf_tables.items():
        print(f"  - {table_name}: {df.shape[0]} rows × {df.shape[1]} columns")
        
    # Show sample of first table
    if pdf_tables:
        first_table_name = list(pdf_tables.keys())[0]
        first_table = pdf_tables[first_table_name]
        
        print(f"\nSample from '{first_table_name}':")
        print(first_table.head())
else:
    print("No PDF tables found")
    print("Note: Make sure you have extracted tables from PDFs in previous parts")
    
    # Create sample data for demonstration
    print("\nCreating sample PDF data for demonstration...")
    sample_pdf_values = {
        'Revenue': 394328000000,  # $394.3B
        'NetIncome': 96995000000,  # $97.0B
        'TotalAssets': 352755000000,  # $352.8B
        'Cash': 29965000000,  # $30.0B
    }
    
    print("Sample PDF financial values:")
    for concept, value in sample_pdf_values.items():
        print(f"  {concept}: ${value/1000000:.1f}M")

# ================================
# CELL 8: Extract Financial Values from PDF Tables
# ================================

if pdf_tables:
    # Extract financial values from PDF tables
    pdf_financial_values = validator_system.validator.extract_financial_values_from_tables(pdf_tables)
    
    print("Financial Values Extracted from PDF Tables:")
    print("=" * 50)
    
    if pdf_financial_values:
        pdf_df = pd.DataFrame([
            {"Financial Concept": concept, "PDF Value ($M)": f"${value/1000000:.1f}M"}
            for concept, value in pdf_financial_values.items()
        ])
        
        print(pdf_df.to_string(index=False))
    else:
        print("No financial values extracted from PDF tables")
        print("This may indicate:")
        print("  - Tables don't contain financial data")
        print("  - Pattern matching needs improvement")
        print("  - Table structure is complex")
else:
    # Use sample data
    pdf_financial_values = sample_pdf_values

# ================================
# CELL 9: Cross-Validation Analysis
# ================================

if 'annual_values' in locals() and 'pdf_financial_values' in locals():
    print("Cross-Validation: PDF vs XBRL")
    print("=" * 50)
    
    # Perform validation
    validation_results = validator_system.validator.validate_against_xbrl(
        pdf_financial_values, annual_values
    )
    
    # Display summary
    summary = validation_results['summary']
    print(f"Validation Summary:")
    print(f"  Total Comparisons: {summary['total_comparisons']}")
    print(f"  Matches: {summary['matches']} ({summary['match_rate_percent']:.1f}%)")
    print(f"  Mismatches: {summary['mismatches']}")
    print(f"  PDF Only: {summary['pdf_only_count']}")
    print(f"  XBRL Only: {summary['xbrl_only_count']}")
    
    # Create comparison DataFrame
    comparison_data = []
    
    # Add matches
    for match in validation_results['matches']:
        comparison_data.append({
            'Concept': match['concept'],
            'PDF Value ($M)': f"${match['pdf_value']/1000000:.1f}M",
            'XBRL Value ($M)': f"${match['xbrl_value']/1000000:.1f}M",
            'Difference (%)': f"{match['percent_difference']:.1f}%",
            'Status': '✓ Match'
        })
    
    # Add mismatches
    for mismatch in validation_results['mismatches']:
        comparison_data.append({
            'Concept': mismatch['concept'],
            'PDF Value ($M)': f"${mismatch['pdf_value']/1000000:.1f}M",
            'XBRL Value ($M)': f"${mismatch['xbrl_value']/1000000:.1f}M",
            'Difference (%)': f"{mismatch['percent_difference']:.1f}%",
            'Status': '✗ Mismatch'
        })
    
    if comparison_data:
        comparison_df = pd.DataFrame(comparison_data)
        print(f"\nDetailed Comparison:")
        print(comparison_df.to_string(index=False))
    
else:
    print("Insufficient data for validation")

# ================================
# CELL 10: Visualization
# ================================

if 'validation_results' in locals() and validation_results['summary']['total_comparisons'] > 0:
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'XBRL vs PDF Validation Results: {TICKER} {YEAR}', fontsize=16)
    
    # 1. Match Rate Pie Chart
    match_data = [
        validation_results['summary']['matches'],
        validation_results['summary']['mismatches']
    ]
    labels = ['Matches', 'Mismatches']
    colors = ['#2ecc71', '#e74c3c']
    
    axes[0, 0].pie(match_data, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    axes[0, 0].set_title('Match Rate')
    
    # 2. Value Comparison Scatter Plot
    if validation_results['matches'] or validation_results['mismatches']:
        pdf_vals = []
        xbrl_vals = []
        concepts = []
        
        for item in validation_results['matches'] + validation_results['mismatches']:
            pdf_vals.append(item['pdf_value'] / 1000000)  # Convert to millions
            xbrl_vals.append(item['xbrl_value'] / 1000000)
            concepts.append(item['concept'])
        
        axes[0, 1].scatter(pdf_vals, xbrl_vals, alpha=0.7, s=100)
        
        # Add perfect correlation line
        min_val = min(min(pdf_vals), min(xbrl_vals))
        max_val = max(max(pdf_vals), max(xbrl_vals))
        axes[0, 1].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8, label='Perfect Match')
        
        axes[0, 1].set_xlabel('PDF Values ($M)')
        axes[0, 1].set_ylabel('XBRL Values ($M)')
        axes[0, 1].set_title('PDF vs XBRL Values')
        axes[0, 1].legend()
        
        # Add concept labels
        for i, concept in enumerate(concepts):
            axes[0, 1].annotate(concept, (pdf_vals[i], xbrl_vals[i]), 
                               xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    # 3. Percentage Differences
    if validation_results['matches'] or validation_results['mismatches']:
        diff_data = [item['percent_difference'] for item in validation_results['matches'] + validation_results['mismatches']]
        concept_names = [item['concept'] for item in validation_results['matches'] + validation_results['mismatches']]
        
        bars = axes[1, 0].bar(range(len(diff_data)), diff_data, 
                             color=['green' if d <= 5 else 'orange' if d <= 10 else 'red' for d in diff_data])
        axes[1, 0].set_xlabel('Financial Concepts')
        axes[1, 0].set_ylabel('Percentage Difference (%)')
        axes[1, 0].set_title('Percentage Differences')
        axes[1, 0].set_xticks(range(len(concept_names)))
        axes[1, 0].set_xticklabels(concept_names, rotation=45, ha='right')
        
        # Add tolerance line
        axes[1, 0].axhline(y=5, color='red', linestyle='--', alpha=0.7, label='5% Tolerance')
        axes[1, 0].legend()
    
    # 4. Coverage Analysis
    coverage_data = [
        validation_results['summary']['total_comparisons'],
        validation_results['summary']['pdf_only_count'], 
        validation_results['summary']['xbrl_only_count']
    ]
    coverage_labels = ['Common', 'PDF Only', 'XBRL Only']
    coverage_colors = ['#3498db', '#f39c12', '#9b59b6']
    
    axes[1, 1].bar(coverage_labels, coverage_data, color=coverage_colors)
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_title('Data Coverage')
    
    plt.tight_layout()
    plt.show()
    
    # Save the plot
    plot_file = Path(OUTPUT_DIR) / f"{TICKER}_{FILING_TYPE}_{YEAR}_validation_plot.png"
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    print(f"\nVisualization saved to: {plot_file}")

# ================================
# CELL 11: Generate Comprehensive Report
# ================================

if 'validation_results' in locals():
    print("Generating comprehensive validation report...")
    
    # Generate report
    report = validator_system.validator.generate_validation_report(
        validation_results, TICKER, str(YEAR)
    )
    
    # Save report
    from datetime import datetime
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    report_file = Path(OUTPUT_DIR) / f"{TICKER}_{FILING_TYPE}_{YEAR}_validation_report_{timestamp}.md"
    
    with open(report_file, 'w') as f:
        f.write(report)
    
    print(f"Report saved to: {report_file}")
    
    # Display key findings
    print(f"\nKey Findings:")
    print(f"=" * 30)
    
    match_rate = validation_results['summary']['match_rate_percent']
    if match_rate >= 80:
        print("🟢 EXCELLENT: High validation accuracy")
    elif match_rate >= 60:
        print("🟡 GOOD: Moderate validation accuracy")
    else:
        print("🔴 NEEDS IMPROVEMENT: Low validation accuracy")
    
    print(f"Match Rate: {match_rate:.1f}%")
    
    if validation_results['mismatches']:
        print(f"\nLargest Discrepancies:")
        sorted_mismatches = sorted(validation_results['mismatches'], 
                                 key=lambda x: x['percent_difference'], reverse=True)
        for i, mismatch in enumerate(sorted_mismatches[:3]):
            print(f"  {i+1}. {mismatch['concept']}: {mismatch['percent_difference']:.1f}% difference")

# ================================
# CELL 12: Automated Mapping Analysis
# ================================

print("\nAnalyzing Automated Mapping Opportunities...")

# Function to calculate text similarity
def text_similarity(text1, text2):
    from difflib import SequenceMatcher
    return SequenceMatcher(None, text1.lower(), text2.lower()).ratio()

# Analyze unmapped concepts
if 'pdf_financial_values' in locals() and 'annual_values' in locals():
    pdf_only_concepts = set(pdf_financial_values.keys()) - set(annual_values.keys())
    xbrl_only_concepts = set(annual_values.keys()) - set(pdf_financial_values.keys())
    
    print(f"Unmapped Concepts Analysis:")
    print(f"  PDF Only: {len(pdf_only_concepts)}")
    print(f"  XBRL Only: {len(xbrl_only_concepts)}")
    
    # Suggest potential mappings based on text similarity
    if pdf_only_concepts and xbrl_only_concepts:
        print(f"\nPotential Mappings (based on text similarity):")
        
        mapping_suggestions = []
        for pdf_concept in pdf_only_concepts:
            best_match = None
            best_score = 0
            
            for xbrl_concept in xbrl_only_concepts:
                score = text_similarity(pdf_concept, xbrl_concept)
                if score > best_score:
                    best_score = score
                    best_match = xbrl_concept
            
            if best_score > 0.3:  # Minimum similarity threshold
                mapping_suggestions.append({
                    'pdf_concept': pdf_concept,
                    'xbrl_concept': best_match,
                    'similarity': best_score
                })
        
        if mapping_suggestions:
            mapping_df = pd.DataFrame(mapping_suggestions)
            mapping_df = mapping_df.sort_values('similarity', ascending=False)
            print(mapping_df.to_string(index=False))
        else:
            print("No high-confidence mappings found")

# ================================
# CELL 13: Summary and Next Steps
# ================================

print("\nXBRL-PDF Validation Analysis Complete!")
print("=" * 50)

print("\nWhat we accomplished:")
print("✓ Downloaded XBRL files from SEC EDGAR")
print("✓ Parsed financial concepts from XBRL data")
print("✓ Compared with PDF-extracted values")
print("✓ Generated validation metrics and visualizations")
print("✓ Created comprehensive report")
print("✓ Identified mapping opportunities")

if 'validation_results' in locals():
    match_rate = validation_results['summary']['match_rate_percent']
    print(f"\nOverall Validation Quality: {match_rate:.1f}%")
    
    print(f"\nNext Steps for Improvement:")
    if match_rate < 50:
        print("1. 🔧 Review PDF table extraction - low match rate suggests systematic issues")
        print("2. 📊 Improve OCR accuracy and table parsing algorithms")
        print("3. 🗺️ Enhance concept mapping between PDF labels and XBRL taxonomy")
    elif match_rate < 80:
        print("1. 🎯 Focus on largest discrepancies identified in the report")
        print("2. 🗺️ Implement automated mapping suggestions")
        print("3. 📊 Fine-tune extraction for specific financial concepts")
    else:
        print("1. ✨ System performing well - focus on edge cases")
        print("2. 📈 Consider expanding to more filings and companies")
        print("3. 🤖 Implement automated monitoring for ongoing validation")

print(f"\nFiles generated:")
if 'report_file' in locals():
    print(f"  - Validation report: {report_file}")
if 'plot_file' in locals():
    print(f"  - Visualization: {plot_file}")

print(f"\nThis analysis provides a foundation for:")
print("  - Measuring PDF extraction quality")
print("  - Identifying systematic parsing errors") 
print("  - Building automated validation pipelines")
print("  - Ensuring financial data accuracy")